# Chapter 5: Rotations Without Matrices

## Rotors, the sandwich product, composition, and inversion

In standard linear algebra, a rotation in $\mathbb{R}^k$ is an orthogonal matrix $U \in SO(k)$ — a $k \times k$ array of numbers satisfying $U^T U = I$ and $\det U = +1$. This works computationally, but it is **opaque geometrically**: looking at the entries of a $256 \times 256$ orthogonal matrix tells you almost nothing about *what* is being rotated or *where*.

Geometric algebra offers a radically different representation. A rotation is encoded by a **rotor**:

$$R = \exp\!\left(-\frac{B\,\theta}{2}\right)$$

where:
- $B$ is a **bivector** — it names the *plane* of rotation (not an axis!).
- $\theta$ is the **angle** of rotation.

The rotor is a single object that bundles the *what* (which plane) and the *how much* (what angle) into one entity. It acts on vectors through the **sandwich product**:

$$v' = R\,v\,R^{-1}$$

which rotates $v$ by angle $\theta$ in the plane specified by $B$.

**Why does this matter for transformers?** Each layer transition in a transformer contains a rotation. The matrix $U$ from polar decomposition is just a rotor in disguise. By extracting the bivector generator $B = \log(U)$, we learn *which planes* the transformer rotates representations through — information that is invisible in the matrix representation.

By the end of this chapter you will be able to:
- Convert an orthogonal matrix into a rotor and read off its bivector generator
- Apply the sandwich product and verify that it preserves norms
- Compose rotors (combine rotations) and invert them (undo rotations)
- Understand why rotor composition in different planes produces richer structure than same-plane composition

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
from scipy.linalg import expm, logm

from layer_time_ga.algebra import (
    Rotor,
    Bivector,
    rotor_from_orthogonal,
    rotor_compose,
    rotor_inverse,
    bivector_from_skew,
)

np.set_printoptions(precision=4, suppress=True)
print("Imports OK")
print(f"NumPy {np.__version__}")

## From Orthogonal Matrix to Rotor

Every rotation matrix $U \in SO(k)$ has a **matrix logarithm**:

$$A = \log(U), \qquad A^T = -A \quad \text{(skew-symmetric)}.$$

The skew-symmetric matrix $A$ *is* the bivector generator $B$ (up to conventions). It encodes:

1. **Which planes** the rotation acts in — each pair of imaginary eigenvalues $\pm i\sigma_j$ defines a principal rotation plane.
2. **How much** rotation occurs in each plane — the magnitude $\sigma_j$ is the rotation angle in that plane.

The conversion pipeline is:

$$U \;\xrightarrow{\;\log\;}\; A \;\xrightarrow{\;\text{skew enforce}\;}\; B \quad\text{(bivector)}$$

The rotor is just the orthogonal matrix wearing a GA hat. The **bivector is the new information** — it tells you the geometry that the matrix hid.

In high dimensions ($k = 256$ is typical for our whitened transformer states), a single rotation generally acts in *multiple planes simultaneously*. The principal plane decomposition via eigenvalues of $B$ reveals this multi-plane structure.

In [ ]:
# ── Build a random rotation in R^8 ──────────────────────────────
# Step 1: Generate a random skew-symmetric matrix (= random bivector)
k = 8
rng = np.random.default_rng(42)
M = rng.standard_normal((k, k))
A_skew = 0.5 * (M - M.T)  # skew-symmetric: A^T = -A

# Step 2: Exponentiate to get an orthogonal matrix (rotation)
U = expm(A_skew)  # U = exp(A), U^T U = I, det(U) = +1

# Verify it is indeed orthogonal
print("U^T U ≈ I?  Max deviation:", np.max(np.abs(U.T @ U - np.eye(k))))
print("det(U) =", np.linalg.det(U))

# Step 3: Convert to a Rotor (this computes the bivector via logm internally)
R = rotor_from_orthogonal(U, compute_bivector=True)

print(f"\n── Rotor Summary ──")
print(f"  Dimension:       {R.dim}")
print(f"  Rotation angle:  {R.angle:.4f} rad  ({np.degrees(R.angle):.1f}°)")
print(f"  Bivector norm:   {R.bivector.norm:.4f}")
print(f"  Is identity?     {R.is_identity}")

# Step 4: Examine the principal planes
planes = R.bivector.principal_planes(n_planes=4)
print(f"\n── Principal Rotation Planes ──")
for i, p in enumerate(planes):
    print(f"  Plane {i+1}: angle = {p['angle']:.4f} rad ({np.degrees(p['angle']):.1f}°),  "
          f"weight = {p['weight']:.4f}")

## The Sandwich Product

In GA, a rotor acts on a vector $v$ through the **sandwich product**:

$$v' = R\,v\,R^{-1}$$

This has two crucial properties:

1. **Norm preservation**: $\|v'\| = \|v\|$. The sandwich product is an isometry — it never stretches or shrinks.
2. **Plane selectivity**: Components of $v$ that lie *in* the rotation plane get rotated by angle $\theta$. Components *perpendicular* to the rotation plane are left **completely unchanged**.

In the matrix representation, the sandwich product reduces to ordinary matrix-vector multiplication: $v' = U v$. The GA perspective adds semantic clarity — we know *why* certain components change and others do not.

**Key insight**: If a transformer layer's rotation acts in a low-dimensional subspace (a few principal planes), then most of the representation is *untouched* by the rotation. The rotation only affects the components that lie within those planes. This is why rotors are a more informative description than raw orthogonal matrices.

In [ ]:
# ── Apply rotor to a vector and verify norm preservation ────────
v = rng.standard_normal(k)
v_rotated = R.apply(v)

print("Original vector v:")
print(f"  {v}")
print(f"  ||v|| = {np.linalg.norm(v):.6f}")

print(f"\nRotated vector v' = R v R^{{-1}}:")
print(f"  {v_rotated}")
print(f"  ||v'|| = {np.linalg.norm(v_rotated):.6f}")

print(f"\nNorm preserved? |  ||v|| - ||v'||  | = {abs(np.linalg.norm(v) - np.linalg.norm(v_rotated)):.2e}")

# ── Show that perpendicular components are unchanged ───────────
# Get the top principal plane
top_plane = planes[0]
u1, u2 = top_plane['plane_vectors']

# Project v onto the rotation plane
v_in_plane = np.dot(v, u1) * u1 + np.dot(v, u2) * u2

# Project v onto the orthogonal complement of the plane
v_perp = v - v_in_plane

# Apply rotor to each component
v_perp_rotated = R.apply(v_perp)
v_in_plane_rotated = R.apply(v_in_plane)

# The perpendicular component should be approximately unchanged
# (not exactly, because R acts in multiple planes, but the top plane
# dominates, so the component perpendicular to ALL planes is unchanged)
print(f"\n── Plane Selectivity (top principal plane) ──")
print(f"  ||v_in_plane||        = {np.linalg.norm(v_in_plane):.4f}")
print(f"  ||v_in_plane_rotated||= {np.linalg.norm(v_in_plane_rotated):.4f}  (norm preserved)")
print(f"  ||v_perp||            = {np.linalg.norm(v_perp):.4f}")
print(f"  ||v_perp_rotated||    = {np.linalg.norm(v_perp_rotated):.4f}  (norm preserved)")
print(f"\n  Change in v_in_plane: {np.linalg.norm(v_in_plane_rotated - v_in_plane):.4f}")
print(f"  Change in v_perp:     {np.linalg.norm(v_perp_rotated - v_perp):.4f}")
print(f"  (Perpendicular component changes less because the rotation primarily acts in the plane)")

## A Concrete Example: 45° in the $e_1 \wedge e_2$ Plane

Let us build a rotor from scratch for a single, specific rotation: **45 degrees in the $e_1$-$e_2$ plane**.

The bivector for the $e_1 \wedge e_2$ plane is the skew-symmetric matrix with $B_{12} = 1$, $B_{21} = -1$, and zeros elsewhere. The rotor matrix is:

$$U = \exp(A), \qquad A = \theta \cdot \begin{pmatrix} 0 & -1 \\ 1 & 0 \\ & & 0 \\ & & & \ddots \end{pmatrix}$$

Applied to $e_1 = (1, 0, 0, \ldots)$, the result should be:

$$e_1' = \cos(45°)\,e_1 + \sin(45°)\,e_2 = \left(\frac{\sqrt{2}}{2},\; \frac{\sqrt{2}}{2},\; 0,\; \ldots\right)$$

This is the simplest possible rotor — a **simple rotor** acting in exactly one plane. In a transformer, layer rotors are generally *compound rotors* acting in many planes simultaneously.

In [ ]:
# ── Build a 45° rotor in the e1-e2 plane of R^8 ───────────────
theta = np.pi / 4  # 45 degrees

# The bivector generator for the e1^e2 plane (skew-symmetric)
A_simple = np.zeros((k, k))
A_simple[0, 1] = -theta   # A_{12} = -θ
A_simple[1, 0] =  theta   # A_{21} = +θ

# The bivector
B_simple = bivector_from_skew(A_simple)
print(f"Bivector for 45° in e1^e2:")
print(f"  B.norm  = {B_simple.norm:.4f}")
print(f"  B.angle = {B_simple.angle:.4f} rad = {np.degrees(B_simple.angle):.1f}°")

# The orthogonal matrix (rotor in matrix form)
U_simple = expm(A_simple)
R_simple = rotor_from_orthogonal(U_simple, compute_bivector=True)

# Apply to e1
e1 = np.zeros(k)
e1[0] = 1.0

e1_rotated = R_simple.apply(e1)

# Expected result
e1_expected = np.zeros(k)
e1_expected[0] = np.cos(theta)
e1_expected[1] = np.sin(theta)

print(f"\ne1         = {e1}")
print(f"R(e1)      = {e1_rotated}")
print(f"Expected   = {e1_expected}")
print(f"Match?       Max error = {np.max(np.abs(e1_rotated - e1_expected)):.2e}")

# Also apply to e3 — it should be completely unchanged
e3 = np.zeros(k)
e3[2] = 1.0
e3_rotated = R_simple.apply(e3)
print(f"\ne3         = {e3}")
print(f"R(e3)      = {e3_rotated}")
print(f"Unchanged?   Max error = {np.max(np.abs(e3_rotated - e3)):.2e}")
print(f"  → e3 is perpendicular to the e1^e2 plane, so it is completely fixed.")

## Composition: Combining Rotations

One of the great advantages of the rotor formalism is that **composition is just multiplication**:

$$R_{12} = R_1 \cdot R_2$$

This means "apply $R_2$ first, then $R_1$." The result $R_{12}$ is itself a rotor — rotations are closed under composition.

Two important cases:

### Same plane
If both rotors act in the *same* plane, the angles simply **add**:
$$R(\theta_1) \cdot R(\theta_2) = R(\theta_1 + \theta_2)$$
The resulting bivector is in the same plane, with the combined angle. This is just like adding angles in 2D rotation.

### Different planes
If the rotors act in *different* planes, the resulting bivector **mixes both planes**. The composed rotor generally acts in a higher-dimensional subspace than either factor. This is where high-dimensional rotation becomes richer than 2D or 3D intuition suggests.

In a transformer, successive layer rotors act in different planes — their composition builds up a complex, multi-plane rotation that cannot be reduced to rotation around a single axis.

In [ ]:
# ════════════════════════════════════════════════════════════════
# CASE 1: Compose two rotors in the SAME plane (e1^e2)
# ════════════════════════════════════════════════════════════════
theta1 = np.pi / 6   # 30°
theta2 = np.pi / 4   # 45°

# Build two rotors in the e1^e2 plane
A1 = np.zeros((k, k))
A1[0, 1] = -theta1;  A1[1, 0] = theta1
R1 = rotor_from_orthogonal(expm(A1), compute_bivector=True)

A2 = np.zeros((k, k))
A2[0, 1] = -theta2;  A2[1, 0] = theta2
R2 = rotor_from_orthogonal(expm(A2), compute_bivector=True)

# Compose: R12 = R1 * R2
R12_same = rotor_compose(R1, R2)
R12_same = rotor_from_orthogonal(R12_same.matrix, compute_bivector=True)

print("── Same Plane Composition ──")
print(f"  R1 angle: {R1.angle:.4f} rad ({np.degrees(R1.angle):.1f}°)")
print(f"  R2 angle: {R2.angle:.4f} rad ({np.degrees(R2.angle):.1f}°)")
print(f"  R12 angle: {R12_same.angle:.4f} rad ({np.degrees(R12_same.angle):.1f}°)")
print(f"  θ1 + θ2  = {theta1 + theta2:.4f} rad ({np.degrees(theta1 + theta2):.1f}°)")
print(f"  Angles add? Error = {abs(R12_same.angle - (theta1 + theta2)):.2e}")

# Verify the composed rotor still acts in the e1^e2 plane
planes_12 = R12_same.bivector.principal_planes(n_planes=2)
print(f"\n  Principal planes of R12:")
for i, p in enumerate(planes_12):
    print(f"    Plane {i+1}: angle = {p['angle']:.4f}, weight = {p['weight']:.4f}")

# ════════════════════════════════════════════════════════════════
# CASE 2: Compose two rotors in DIFFERENT planes
# ════════════════════════════════════════════════════════════════
print("\n── Different Plane Composition ──")

# R_a: 30° in e1^e2 plane (already built as R1)
# R_b: 45° in e3^e4 plane
A_b = np.zeros((k, k))
A_b[2, 3] = -theta2;  A_b[3, 2] = theta2
R_b = rotor_from_orthogonal(expm(A_b), compute_bivector=True)

# Compose
R_ab = rotor_compose(R1, R_b)
R_ab = rotor_from_orthogonal(R_ab.matrix, compute_bivector=True)

print(f"  R_a: {R1.angle:.4f} rad in e1^e2")
print(f"  R_b: {R_b.angle:.4f} rad in e3^e4")
print(f"  R_ab angle: {R_ab.angle:.4f} rad ({np.degrees(R_ab.angle):.1f}°)")

# The composed bivector now has BOTH planes
planes_ab = R_ab.bivector.principal_planes(n_planes=4)
print(f"\n  Principal planes of R_ab (both planes present!):")
for i, p in enumerate(planes_ab):
    print(f"    Plane {i+1}: angle = {p['angle']:.4f}, weight = {p['weight']:.4f}")

# Verify: bivector has nonzero components in both e1^e2 and e3^e4
B_ab = R_ab.bivector.matrix
print(f"\n  Bivector e1^e2 component (B[0,1]): {B_ab[0,1]:.4f}")
print(f"  Bivector e3^e4 component (B[2,3]): {B_ab[2,3]:.4f}")
print(f"  → The composed rotation spans both planes.")

## Inversion: Undoing a Rotation

Every rotor has an inverse that perfectly undoes the rotation:

$$R^{-1} = R^T \qquad \text{(for orthogonal matrices, transpose = inverse)}$$

In GA language, $R^{-1}$ is the **reversion** $\tilde{R}$: the rotor with the *same plane* but the *opposite angle*. The defining property is:

$$R \cdot R^{-1} = I \qquad \text{(identity rotor = no rotation)}$$

If $R$ has bivector generator $B$, then $R^{-1}$ has generator $-B$ — same planes, negated angles.

This is conceptually clean: to undo a 30° rotation in the $e_1 \wedge e_2$ plane, apply a $-30°$ rotation in the same plane. No matrix inversion needed.

In [ ]:
# ── Demonstrate rotor inversion ────────────────────────────────
# Use the random rotor R from earlier
R_inv = rotor_inverse(R)

print("── Rotor Inversion ──")
print(f"  R.angle      = {R.angle:.4f} rad")
print(f"  R_inv.angle  = {R_inv.angle:.4f} rad")
print(f"  (Same magnitude, opposite direction)")

# Verify R * R^{-1} ≈ I
R_identity = rotor_compose(R, R_inv)
identity_deviation = np.linalg.norm(R_identity.matrix - np.eye(k), 'fro')
print(f"\n  R * R^{{-1}} ≈ I?")
print(f"  ||R R^{{-1}} - I||_F = {identity_deviation:.2e}")

# Verify the bivector generators are negated
if R.bivector is not None and R_inv.bivector is not None:
    B_sum = R.bivector.matrix + R_inv.bivector.matrix
    print(f"\n  B + B_inv ≈ 0?")
    print(f"  ||B + B_inv||_F = {np.linalg.norm(B_sum, 'fro'):.2e}")

# Round-trip: rotate a vector, then unrotate it
v_test = rng.standard_normal(k)
v_rotated_test = R.apply(v_test)
v_back = R_inv.apply(v_rotated_test)

print(f"\n  Round-trip test:")
print(f"  ||v - R^{{-1}}(R(v))||  = {np.linalg.norm(v_back - v_test):.2e}")
print(f"  → Perfect recovery: the inverse exactly undoes the rotation.")

## Three Ways to Build a Rotation

We now have three methods for constructing rotations from their bivector generators.
Each has different trade-offs:

| Method | Formula | Pros | Cons |
|--------|---------|------|------|
| **Matrix logarithm** | $B\theta = \log(U)$ | Exact, handles multiple planes | $O(k^3)$, requires `logm` |
| **Cayley map** | $A = (U-I)(U+I)^{-1}$ | Algebraic (no transcendentals) | Fails at 180° rotations |
| **Rodrigues** | $R = I + \sin\theta\,\hat{B} + (1-\cos\theta)\hat{B}^2$ | Exact, $O(k^2)$ from two vectors | Single plane only |

Use the **matrix logarithm** for layer transition operators (multiple planes).
Use **Rodrigues or Cayley** when comparing two vectors (single plane).

In [ ]:
# ── Compare the three rotation methods ──────────────────────────
from layer_time_ga.algebra import (
    rodrigues_rotation, rodrigues_rotor, cayley_bivector,
    rotor_from_orthogonal, bivector_from_skew
)
from scipy.linalg import expm, logm

# Two random unit vectors in R^8
rng = np.random.default_rng(42)
u = rng.standard_normal(8); u /= np.linalg.norm(u)
v = rng.standard_normal(8); v /= np.linalg.norm(v)
theta = np.arccos(np.clip(np.dot(u, v), -1, 1))

print(f"Angle between u and v: {np.degrees(theta):.1f}°\n")

# Method 1: Rodrigues (direct from two vectors)
R_rod = rodrigues_rotation(u, v)
print(f"Rodrigues:  ||Ru - v|| = {np.linalg.norm(R_rod @ u - v):.2e}")

# Method 2: Cayley map (from the Rodrigues matrix)
A_cay = (R_rod - np.eye(8)) @ np.linalg.inv(R_rod + np.eye(8))
print(f"Cayley map: ||A||_F/√2 = {np.linalg.norm(A_cay, 'fro')/np.sqrt(2):.4f}  "
      f"(tan(θ/2) = {np.tan(theta/2):.4f})")

# Method 3: Matrix logarithm
B_log = logm(R_rod)
print(f"Matrix log: ||B||_F/√2 = {np.linalg.norm(B_log, 'fro')/np.sqrt(2):.4f}  "
      f"(θ = {theta:.4f})")

# The Cayley bivector shortcut (direct from vectors)
biv, tau = cayley_bivector(u, v)
print(f"\nCayley bivector (direct): τ = {tau:.4f}")

# All three give the same rotation plane
rotor_rod = rodrigues_rotor(u, v)
plane_rod = rotor_rod.bivector.principal_planes(1)[0]
print(f"Principal plane angle from Rodrigues rotor: {plane_rod['angle']:.4f} rad")

## In the Transformer

Every layer transition in a transformer contains a rotation. When we compute the polar decomposition $T^{(l)} = U^{(l)} P^{(l)}$, the orthogonal factor $U^{(l)}$ *is* a rotor. The collection of rotors across all layers:

$$\{R^{(0)}, R^{(1)}, \ldots, R^{(L-2)}\}$$

is the **rotor field** on the layer axis. Each rotor carries a bivector generator $B^{(l)} = \log(R^{(l)})$ that tells us the *planes and angles* of rotation at that layer.

Some key observations from real models:
- **Early layers** tend to have larger rotation angles (more geometric rearrangement).
- **Late layers** often show smaller angles (fine-tuning of directions).
- **Different prompts** produce different rotor fields — the geometry adapts to the input.

The bivector representation makes it possible to ask questions that are invisible to matrix analysis:
- Do successive layers rotate in the *same* planes or *different* planes?
- How much does the rotation plane *change* from one layer to the next?
- Do the bivectors *commute* ($[B_i, B_j] = 0$) or is there genuine curvature?

These questions lead directly to **Chapter 6**, where we decompose what each layer actually does into its rotation (grade-2) and stretching (grade-0) components.

## Exercises

**Exercise 5.1 — Norm preservation under composition.**
Compose three rotors acting in three different planes ($e_1 \wedge e_2$, $e_3 \wedge e_4$, $e_5 \wedge e_6$) with angles 20°, 35°, and 50°. Apply the composed rotor to 100 random vectors and verify that every norm is preserved to machine precision.

**Exercise 5.2 — Three-rotor composition.**
Build three rotors:
- $R_1$: 30° in $e_1 \wedge e_2$
- $R_2$: 45° in $e_1 \wedge e_2$
- $R_3$: 60° in $e_3 \wedge e_4$

Compute $R_{123} = R_1 \cdot R_2 \cdot R_3$. How many principal planes does the result have? What are their angles? Compare to $R_{132} = R_1 \cdot R_3 \cdot R_2$ — is rotor composition commutative?

**Exercise 5.3 — Angle from rotor vs. Frobenius distance.**
For rotation angles $\theta \in \{5°, 15°, 30°, 60°, 90°, 120°, 150°, 175°\}$, build a rotor in the $e_1 \wedge e_2$ plane and compute both:
1. The rotor angle from `R.angle`
2. The Frobenius distance $\|U - I\|_F$

Plot both quantities against $\theta$. How do they relate? For what range of $\theta$ is $\|U - I\|_F$ a good proxy for the rotation angle?

**Exercise 5.4 — High-dimensional random rotors.**
Generate 50 random rotors in $\mathbb{R}^{64}$ (using random skew-symmetric generators with entries scaled by 0.1). For each, extract the principal planes. Plot a histogram of the top-plane weight as a fraction of total weight. Does a typical random rotation concentrate in a few planes or spread across many?